# 1. Mục tiêu
Dự báo PM2.5 giờ kế tiếp theo trạm. Notebook chỉ giải thích thí nghiệm; pipeline chính nằm trong `src/`.

## 2. Phạm vi và cảnh báo
Đây là mô hình nghiên cứu, không phải hệ thống cảnh báo chính thức.

## 3. Thiết lập môi trường

In [1]:
from pathlib import Path
import sys
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

## 4. Nạp cấu hình và tìm dữ liệu
Đường dẫn hỗ trợ local, thư mục dự án và `/content` trên Colab.

In [2]:
from src.data import load_air_quality, load_config
config = load_config(PROJECT_ROOT / 'configs/config.yaml')
data = load_air_quality(config)
data.head()

,timestamp,station,PM2.5,TSP,NO2,SO2,CO,O3,temperature,humidity
0,2024-01-01 00:00:00,Trạm A,18.00,33.00,12.0,NaN,0.5,NaN,29.00,70.00
1,2024-01-01 01:00:00,Trạm A,19.21,34.21,13.0,4.0,0.6,21.0,29.33,69.29
2,2024-01-01 02:00:00,Trạm A,20.36,35.36,14.0,5.0,0.7,22.0,29.65,68.59
3,2024-01-01 03:00:00,Trạm A,21.39,36.39,15.0,6.0,0.8,23.0,29.96,67.92
4,2024-01-01 04:00:00,Trạm A,22.25,37.25,16.0,3.0,0.9,24.0,30.24,67.30


## 5. Kiểm tra schema và thứ tự thời gian

In [3]:
timestamp_column = config['data']['timestamp_column']
station_column = config['data']['station_column']
assert data.groupby(station_column)[timestamp_column].apply(lambda x: x.is_monotonic_increasing).all()
data.shape

(72, 10)

## 6. Quy tắc giá trị 0
Mặc định giữ 0. Chỉ bật `zero_as_missing` sau khi có bằng chứng hoặc sensitivity analysis.

In [4]:
target_column = config['data']['target_column']
{'pm25_zero_or_less': int((data[target_column] <= 0).sum()), 'zero_as_missing': config['data']['zero_as_missing']}

{'pm25_zero_or_less': 0, 'zero_as_missing': False}

## 7. Tạo đặc trưng chống rò rỉ dữ liệu

In [5]:
from src.features import build_features, model_feature_columns
featured = build_features(data, config)
feature_columns = model_feature_columns(config)
featured.head()

,timestamp,station,PM2.5,TSP,NO2,SO2,CO,O3,temperature,humidity,...,PM2.5_lag_6,PM2.5_lag_12,PM2.5_lag_24,PM2.5_rolling_mean_3,PM2.5_rolling_mean_6,PM2.5_rolling_mean_24,hour_sin,hour_cos,day_of_week,target_next_hour
0,2024-01-01 00:00:00,Trạm A,18.00,33.00,12.0,NaN,0.5,NaN,29.00,70.00,...,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,1.000000,0,19.21
1,2024-01-01 01:00:00,Trạm A,19.21,34.21,13.0,4.0,0.6,21.0,29.33,69.29,...,NaN,NaN,NaN,18.000,18.000,18.000,0.258819,0.965926,0,20.36
2,2024-01-01 02:00:00,Trạm A,20.36,35.36,14.0,5.0,0.7,22.0,29.65,68.59,...,NaN,NaN,NaN,18.605,18.605,18.605,0.500000,0.866025,0,21.39
3,2024-01-01 03:00:00,Trạm A,21.39,36.39,15.0,6.0,0.8,23.0,29.96,67.92,...,NaN,NaN,NaN,19.190,19.190,19.190,0.707107,0.707107,0,22.25
4,2024-01-01 04:00:00,Trạm A,22.25,37.25,16.0,3.0,0.9,24.0,30.24,67.30,...,NaN,NaN,NaN,20.320,19.740,19.740,0.866025,0.500000,0,22.90


## 8. Kiểm chứng lag một giờ

In [6]:
sample_station = featured[featured[station_column] == featured[station_column].iloc[0]]
assert sample_station[f'{target_column}_lag_1'].iloc[1] == sample_station[target_column].iloc[0]

## 9. Chia train/test theo thời gian
Pipeline huấn luyện thực hiện time split, không shuffle.

In [7]:
usable = featured.dropna(subset=['target_next_hour']).sort_values(timestamp_column)
cut = int(len(usable) * (1 - config['split']['test_fraction']))
usable.iloc[:cut][timestamp_column].max(), usable.iloc[cut:][timestamp_column].min()

(Timestamp('2024-01-02 03:00:00'), Timestamp('2024-01-02 04:00:00'))

## 10. Baseline
Nên so sánh persistence baseline và mô hình bằng cùng một time split.

In [8]:
from sklearn.metrics import mean_absolute_error
baseline = usable[target_column]
baseline_mae = mean_absolute_error(usable['target_next_hour'], baseline)
baseline_mae

np.float64(0.6474285714285714)

## 11. Huấn luyện mô hình
Lệnh chuẩn: `python -m src.train --config configs/config.yaml`.

## 12. Đánh giá
Báo cáo MAE, RMSE, Macro-F1, QWK, confusion matrix, kết quả theo lớp và trạm.

In [9]:
from src.evaluate import classify_pm25
set(classify_pm25([5, 20, 50], config['thresholds']['good_max'], config['thresholds']['moderate_max']))

{np.str_('Trung bình'), np.str_('Tốt'), np.str_('Xấu')}

## 13. Ablation đặc trưng
Đoạn dưới đã sửa lỗi cú pháp; output cũ đã được xóa.

In [10]:
numeric_features = feature_columns
pm25_history_features = [
    feature
    for feature in numeric_features
    if feature == 'PM2.5' or feature.startswith('PM2.5_')
]

exogenous_features = [
    feature
    for feature in numeric_features
    if feature != 'PM2.5' and not feature.startswith('PM2.5_')
]

pm25_history_features, exogenous_features

(['PM2.5',
  'PM2.5_lag_1',
  'PM2.5_lag_2',
  'PM2.5_lag_3',
  'PM2.5_lag_6',
  'PM2.5_lag_12',
  'PM2.5_lag_24',
  'PM2.5_rolling_mean_3',
  'PM2.5_rolling_mean_6',
  'PM2.5_rolling_mean_24'],
 ['TSP',
  'NO2',
  'SO2',
  'CO',
  'O3',
  'temperature',
  'humidity',
  'hour_sin',
  'hour_cos',
  'day_of_week'])

## 14. Sensitivity analysis cho giá trị 0
Chạy hai cấu hình giữ 0 và chuyển 0 thành NaN; so sánh MAE, recall lớp Xấu và số lượng bị impute.

## 15. Rolling validation
Cần công bố rolling MAE trung bình ± độ lệch chuẩn; số lịch sử 3,636 ± 1,109 cho thấy biến động đáng kể.

## 16. Kết luận và tái lập
Chạy test, training, API và dashboard theo README. Không dùng điểm từ dữ liệu tổng hợp làm kết quả nghiên cứu.